In [89]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
data=np.genfromtxt("Mus.csv",delimiter=",",skip_header=1)

In [ ]:
def double_Rayleigh(data,sfs,bins=40,minlike=0.8,percentile=99,fps=None,plotting=True):

    # Givet likelihood af om musens kropsdele rent faktisk er de steder hvor vi har optaget dem. n er nose osv. 
    nlike=data[:,3]
    ellike=data[:,6]
    erlike=data[:,9]
    sllike=data[:,12]
    srlike=data[:,15]
    tlike=data[:,18]

    #fjerner frames med dårlig tracking
    # Vi laver en maske som er et numpyarray der giver sand/ falsk om likelihood er over 0.8 for hver kropsdel
    mask = (nlike > minlike)&(ellike > minlike)&(erlike > minlike)&(sllike > minlike)&(srlike > minlike)&(tlike > minlike)
    # Her smider man data ud, men det er kun når vi plotter 

    # Nu sætter vi den maske på alle x og y værdierne for musens kropsdeles position, og funktion gør at den kun der over minlike
    #Men vi benytter kun skuldrenes position og og halens
    x_sl=data[:,10][mask]
    y_sl=data[:,11][mask]
    x_sr=data[:,13][mask]
    y_sr=data[:,14][mask]
    x_t=data[:,16][mask]
    y_t=data[:,17][mask]

    #vi finde nu midten af musen, så vi kan plotte den 
    x_midS = (x_sl + x_sr) / 2
    y_midS = (y_sl + y_sr) / 2
    x_mid = (x_t + x_midS) / 2
    y_mid = (y_t + y_midS) / 2

    #Vi gruppere punkterne i blokke af samples per frame step og kræver at de gpår lige op ved at tage modolus
    n = len(x_midS)%sfs
    if n!=0:
        x_trunc=x_mid[:-n] #Np arrays med (x1,x2....xn)
        y_trunc=y_mid[:-n]#Np arrays med (y1,y2....yn)
    else:
        x_trunc=x_mid
        y_trunc=y_mid

    #reshape(-1, sfs) kræver at antallet af punkter kan deles præcist med sfs.
    x_block=x_trunc.reshape(-1, sfs)
    y_block=y_trunc.reshape(-1, sfs)

    # Vi finder nu hastigheden af musen ved at tage forskellen mellem positionerne i hver blok og dividere med tiden mellem hvert step
    dx = x_block[:, -1] - x_block[:, 0]
    dy = y_block[:, -1] - y_block[:, 0]
    speed = np.sqrt(dx**2 + dy**2)

 

    #Weighted rayleighs

    #Det er en mixture distribution: med sandsynlighed w: Rayleigh(σ1) #med sandsynlighed 1-w: Rayleigh(σ2)

                #Fortolkning i musedata: ofte “to regimes”:
                #langsom “snusen/pauser” (lille σ)
                #hurtig “løb/forflytning” (stor σ)
    def Weighted_Rayleigh_pdf(x,sigma1,sigma2,weight):
        r1 = (x / sigma1**2) * np.exp(-x**2 / (2 * sigma1**2))
        r2 = (x / sigma2**2) * np.exp(-x**2 / (2 * sigma2**2))
        return weight * r1 + (1-weight) * r2


    # Nu skal vi finde de to Rayleigh fordelinger der bedst passer til vores data, og det gør vi ved at minimere den negative log-likelihood

    def neg_log_likelihood(params, data):
        # Her definerer vi den negative log-likelihood funktion
        sigma1, sigma2, weight = params

        # Vi beregner den teoretiske Rayleigh fordeling
        pdf_vals = Weighted_Rayleigh_pdf(data, sigma1, sigma2, weight)
        # Vi beregner den negative log-likelihood
        return -np.sum(np.log(pdf_vals+1e-12))

    p0 = [np.mean(speed)/2, np.mean(speed)*2, 0.5]
    # Vi starter med nogle gæt for sigma1, sigma2 og weight
    p0 = [np.mean(speed)/2, np.mean(speed)*2, 0.5]
    res = minimize(neg_log_likelihood, x0=p0, args=(speed,),
        bounds=[(1e-6, None),(1e-6, None),(1e-3, 1-1e-3)])
    
    sigma1, sigma2, weight = res.x
    Delta_t = 5*10**-3*sfs

    D1 = sigma1**2 / (2 * Delta_t)
    D2 = sigma2**2 / (2 * Delta_t)

    

    if plotting==True:
        plt.figure(figsize=(8,6))
        upper = np.percentile(speed, percentile)
        plt.hist(speed, bins=bins, range=(0, upper), density=True, alpha=0.5, color='C0', label="Histogram")
        X = np.linspace(0, upper, 1000)
        Y = Weighted_Rayleigh_pdf(X, sigma1, sigma2, weight)
        plt.plot(X, Y, 'r', lw=2, label=f"Log-likelihood maximized PDF: σ1={sigma1:.2f}, σ2={sigma2:.2f}, w={weight:.2f}")
        plt.legend()
        plt.xlabel("Hastighed i arbitrær enhed")
        plt.ylabel("PDF")
        plt.title("2 Rayleigh fit")
        plt.show()
    return sigma1, sigma2, weight

print(D1, D2)

double_Rayleigh(data,sfs=50,bins=40,minlike=0.8,percentile=99,plotting=True)

NameError: name 'D1' is not defined

$$ x_{k+1}​=x_k​\sqrt {2DΔt​η_{x,k}}​$$

$$y_{k+1}​=x_k​\sqrt {2DΔt​η_{y,k}}​ $$
hvor ​η er normal fordelt og incrementerne er gaussiske med normal fordeling $2DΔt$


N(0,2DΔt)


$\sigma^2=2DΔt$. => $D=\frac{\sigma^2}{2DΔt}$

For de to rayleighs 
$D1=\frac{\sigma_1^2}{2Δt}$, 

$D2=\frac{\sigma_2^2}{2Δt}$


In [ ]:
#Da vi ikke kender fps sætter vi dt=1
dt=()
